# Explainable Phishing Detection Under Distribution Shift

Hybrid ensembles, calibration, and selective prediction on the UCI Phishing Websites dataset.

This notebook reproduces every number and figure in the repository. It has two parts:

1. **Static study** - seven models (Decision Tree, Random Forest, Logistic Regression, SVM, tuned Random Forest, soft voting, stacking) on a conventional stratified 80/20 split, with SHAP and permutation importance.
2. **Drift study** - a simulated distribution shift that holds the class prior fixed, KS tests, calibration, abstention with a threshold locked on validation data, and SHAP explanation stability.

**Label encoding.** In the UCI file `-1` = phishing and `1` = legitimate. The mapping is fixed in code (never inferred from class fractions) and verified with a sanity check: valid SSL (`SSLfinal_State = 1`) must be *negatively* correlated with phishing. The positive class everywhere is **phishing**.

**On Kaggle:** add the dataset (*Add Input -> search "phishing websites dataset"*, e.g. `mmelgizin/phishing-websites-dataset`) and choose *Run All*. It finds the CSV under `/kaggle/input` automatically. Runtime is roughly 5-10 minutes on CPU.

In [ ]:
%pip install -q shap


## The `phishdrift` package

The cells below write the package source to disk so the rest of the notebook can import it. The same code lives in the GitHub repository.

In [ ]:
import os
os.makedirs('phishdrift', exist_ok=True)

In [ ]:
%%writefile phishdrift/__init__.py
"""phishdrift: explainable hybrid-ensemble phishing detection, evaluated under
distribution shift with calibration, selective prediction and explanation
stability.

Modules
-------
config      constants shared across the pipeline
data        loading, cleaning and (verified) label encoding
models      base classifiers, tuned Random Forest, soft-voting and stacking
drift       simulated drift splits (naive and class-balanced) and KS tests
abstention  calibration, validation-locked abstention threshold, risk-coverage
explain     SHAP attributions and explanation-stability metrics
plots       every figure in the README / paper
"""

__version__ = "1.0.0"


In [ ]:
%%writefile phishdrift/abstention.py
"""Calibration and selective prediction (abstention).

The model answers only when its calibrated confidence max(p, 1-p) reaches a
threshold tau. Tau is chosen on the *validation* partition and locked before
the future partition is ever scored - tuning it on the future data would
report an optimistically biased trade-off.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, log_loss, recall_score

from .config import (CALIBRATION_CV, CALIBRATION_METHOD, MIN_VAL_COVERAGE,
                     TARGET_SELECTIVE_ACCURACY, THRESHOLD_GRID)


def fit_calibrated(base_model, X_train, y_train,
                   method=CALIBRATION_METHOD, cv=CALIBRATION_CV):
    """Cross-fitted calibration on the training partition only."""
    model = CalibratedClassifierCV(base_model, method=method, cv=cv)
    model.fit(X_train, y_train)
    return model


def confidence(prob: np.ndarray) -> np.ndarray:
    return np.maximum(prob, 1 - prob)


def expected_calibration_error(y, prob, n_bins: int = 10) -> float:
    conf = confidence(prob)
    pred = (prob >= 0.5).astype(int)
    edges = np.linspace(0.5, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf < hi) if hi < 1 else (conf >= lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs((pred[m] == np.asarray(y)[m]).mean() - conf[m].mean())
    return float(ece)


def calibration_metrics(y, prob) -> dict:
    return {"brier": float(brier_score_loss(y, prob)),
            "log_loss": float(log_loss(y, np.clip(prob, 1e-6, 1 - 1e-6))),
            "ece": expected_calibration_error(y, prob)}


def select_threshold(y_val, prob_val, target=TARGET_SELECTIVE_ACCURACY,
                     min_coverage=MIN_VAL_COVERAGE, grid=THRESHOLD_GRID) -> float:
    """Lowest tau whose validation selective accuracy reaches ``target`` while
    still answering at least ``min_coverage`` of cases. If no tau reaches the
    target, return the most accurate tau that respects the coverage floor."""
    y_val = np.asarray(y_val)
    conf = confidence(prob_val)
    pred = (prob_val >= 0.5).astype(int)
    best_tau, best_acc = grid[0], -1.0
    for tau in grid:
        m = conf >= tau
        cov = m.mean()
        if cov < min_coverage or not m.any():
            continue
        acc = (pred[m] == y_val[m]).mean()
        if acc >= target:
            return float(tau)
        if acc > best_acc:
            best_tau, best_acc = tau, acc
    return float(best_tau)


def selective_metrics(y, prob, tau: float) -> dict:
    y = np.asarray(y)
    pred = (prob >= 0.5).astype(int)
    answered = confidence(prob) >= tau
    out = {
        "accuracy_all": float((pred == y).mean()),
        "coverage": float(answered.mean()),
        "abstention_rate": float(1 - answered.mean()),
        "selective_accuracy": float((pred[answered] == y[answered]).mean()) if answered.any() else float("nan"),
        "recall_phishing_all": float(recall_score(y, pred, pos_label=1, zero_division=0)),
        "recall_legitimate_all": float(recall_score(y, pred, pos_label=0, zero_division=0)),
    }
    if answered.any():
        out["recall_phishing_answered"] = float(recall_score(y[answered], pred[answered], pos_label=1, zero_division=0))
        out["recall_legitimate_answered"] = float(recall_score(y[answered], pred[answered], pos_label=0, zero_division=0))
    return out


def risk_coverage(y, prob, n_points: int = 50) -> pd.DataFrame:
    """Error rate among the most-confident ``coverage`` fraction of cases."""
    y = np.asarray(y)
    order = np.argsort(-confidence(prob), kind="mergesort")
    correct = ((prob >= 0.5).astype(int) == y)[order]
    rows = []
    for c in np.linspace(0.05, 1.0, n_points):
        n = max(1, int(round(c * len(y))))
        rows.append({"coverage": n / len(y), "risk": 1 - correct[:n].mean()})
    return pd.DataFrame(rows)


def confidence_bins(y, prob, edges=(0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0001)) -> pd.DataFrame:
    """Accuracy within confidence bands - is confidence still diagnostic?"""
    y = np.asarray(y)
    conf = confidence(prob)
    pred = (prob >= 0.5).astype(int)
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf < hi)
        rows.append({"bin": f"[{lo:.2f}, {min(hi, 1):.2f}{']' if hi > 1 else ')'}",
                     "n": int(m.sum()), "share": float(m.mean()),
                     "accuracy": float((pred[m] == y[m]).mean()) if m.any() else float("nan")})
    return pd.DataFrame(rows)


In [ ]:
%%writefile phishdrift/config.py
"""Constants shared across the pipeline.

Everything that affects a reported number lives here, so a reader can see
the whole experimental setup in one place.
"""

RANDOM_STATE = 42

# ---- static study -----------------------------------------------------------
TEST_SIZE = 0.20          # held-out fraction for the conventional split
CV_FOLDS = 5              # stratified folds for CV-F1, grid search, stacking

RF_GRID = {
    "n_estimators": [200, 400],
    "max_depth": [None, 20, 30],
    "min_samples_leaf": [1, 2],
}

# ---- drift study ------------------------------------------------------------
FUTURE_FRACTION = 0.30    # share of each class assigned to the "future" partition
VAL_FRACTION = 0.20       # share of the remaining "old" pool used for validation
N_DRIFT_FEATURES = 10     # most discriminative features used to score atypicality
KS_ALPHA = 0.05           # significance level for per-feature KS tests

# ---- calibration / abstention ----------------------------------------------
CALIBRATION_METHOD = "isotonic"
CALIBRATION_CV = 5
TARGET_SELECTIVE_ACCURACY = 0.99   # accuracy the threshold must reach on validation
MIN_VAL_COVERAGE = 0.50            # never accept a threshold that answers < 50%
THRESHOLD_GRID = [round(0.50 + 0.01 * i, 2) for i in range(50)]  # 0.50 .. 0.99

# ---- explanation stability --------------------------------------------------
SHAP_SAMPLE = 800         # rows per partition explained with SHAP
TOP_K = 10                # Jaccard@k overlap of top-k features

# ---- labels -----------------------------------------------------------------
# UCI Phishing Websites encodes the target as  -1 = phishing, 1 = legitimate.
# Verified against the dataset's published composition (4,898 phishing /
# 6,157 legitimate of 11,055) and against the raw value counts of the file.
# The mapping is fixed here on purpose: it must NOT be inferred from class
# fractions, because deduplication changes those fractions.
PHISHING_RAW_VALUE = -1
CLASS_NAMES = ["legitimate", "phishing"]   # index 0, index 1 (positive class)

# ---- plotting ---------------------------------------------------------------
# Fixed categorical order (colorblind-validated): blue, orange, aqua, yellow.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
BLUE, ORANGE, AQUA, YELLOW = SERIES
INK, MUTED, GRID = "#1f1f1e", "#6b6a64", "#e4e3de"
SEQ_CMAP = "Blues"        # single-hue sequential for confusion matrices


In [ ]:
%%writefile phishdrift/data.py
"""Loading, cleaning and label encoding for the UCI Phishing Websites corpus.

The label direction is the single easiest thing to get silently wrong in this
dataset, so it is hard-coded (see ``config.PHISHING_RAW_VALUE``) and then
checked against domain knowledge: a valid SSL certificate must push a site
*toward* legitimate. If the encoded labels disagree, loading fails loudly
instead of producing inverted results.
"""

from __future__ import annotations

import glob
import os
from dataclasses import dataclass, field

import numpy as np
import pandas as pd

from .config import PHISHING_RAW_VALUE

NON_FEATURE_COLUMNS = ["index", "Index", "id", "ID", "Unnamed: 0",
                       "FILENAME", "URL", "url", "Domain", "domain"]
LABEL_CANDIDATES = ["Result", "result", "class", "Class", "label", "Label",
                    "target", "Target", "status"]
SEARCH_ROOTS = ["/kaggle/input", "data", "../data", "."]


@dataclass
class DataInfo:
    path: str
    raw_rows: int
    label_column: str
    raw_counts: dict
    dropped_columns: list
    duplicates_removed: int
    conflicting_feature_vectors: int
    rows: int
    n_features: int
    phishing_rate: float
    ssl_label_correlation: float | None
    notes: list = field(default_factory=list)


def find_dataset(path: str | None = None) -> str:
    """Return a path to the dataset, searching the usual places if needed."""
    if path and os.path.exists(path):
        return path
    for root in SEARCH_ROOTS:
        for pattern in ("**/*.csv", "**/*.arff"):
            hits = sorted(glob.glob(os.path.join(root, pattern), recursive=True))
            hits = [h for h in hits if "results" not in h]
            if hits:
                return hits[0]
    raise FileNotFoundError(
        "Dataset not found. Download the UCI Phishing Websites dataset and put "
        "the CSV in data/ (see data/README.md), or pass --data PATH.")


def load_raw(path: str) -> pd.DataFrame:
    """Read the CSV, or the original UCI ARFF file."""
    if path.lower().endswith(".arff"):
        from scipy.io import arff
        records, _ = arff.loadarff(path)
        df = pd.DataFrame(records)
        for col in df.columns:
            if df[col].dtype == object:
                df[col] = df[col].str.decode("utf-8").astype(int)
        return df
    return pd.read_csv(path)


def _label_column(df: pd.DataFrame) -> str:
    for c in LABEL_CANDIDATES:
        if c in df.columns:
            return c
    return df.columns[-1]


def check_label_direction(X: pd.DataFrame, y: pd.Series) -> float | None:
    """Fail if phishing labels look inverted.

    ``SSLfinal_State`` = 1 means a trusted, valid certificate. It must be
    negatively correlated with the phishing label. A positive correlation
    means the labels are flipped and every downstream number (F1, precision,
    recall, SHAP directions, class priors) would be reported for the wrong
    class.
    """
    if "SSLfinal_State" not in X.columns:
        return None
    corr = float(np.corrcoef(X["SSLfinal_State"], y)[0, 1])
    if corr > 0:
        raise ValueError(
            f"Label direction check failed: corr(SSLfinal_State, phishing) = "
            f"{corr:+.3f} > 0, i.e. valid SSL would predict phishing. The "
            f"labels are inverted - check config.PHISHING_RAW_VALUE.")
    return corr


def prepare(df_raw: pd.DataFrame, path: str = "", verbose: bool = True):
    """Clean the raw frame and return ``(X, y, info)``.

    Steps: drop identifier columns, drop exact duplicate rows, encode the
    label with the fixed mapping (1 = phishing), coerce features to numeric,
    drop constant columns, then verify the label direction.
    """
    df = df_raw.copy()
    raw_rows = len(df)
    dropped = [c for c in NON_FEATURE_COLUMNS if c in df.columns]
    df = df.drop(columns=dropped)

    label_col = _label_column(df)
    raw_counts = {str(k): int(v) for k, v in df[label_col].value_counts().items()}

    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    duplicates_removed = before - len(df)

    features = [c for c in df.columns if c != label_col]
    conflicting = int(df.duplicated(subset=features, keep=False).sum())

    values = set(pd.unique(df[label_col]))
    if not values <= {-1, 1, 0}:
        raise ValueError(f"Unexpected label values {sorted(values)}")
    y = (df[label_col] == PHISHING_RAW_VALUE).astype(int).rename("phishing")

    X = df[features].apply(pd.to_numeric, errors="coerce")
    if X.isna().any().any():
        X = X.fillna(X.median())
    constant = X.columns[X.nunique() <= 1].tolist()
    X = X.drop(columns=constant)

    corr = check_label_direction(X, y)

    info = DataInfo(
        path=path, raw_rows=raw_rows, label_column=label_col,
        raw_counts=raw_counts, dropped_columns=dropped + constant,
        duplicates_removed=duplicates_removed,
        conflicting_feature_vectors=conflicting, rows=len(X),
        n_features=X.shape[1], phishing_rate=float(y.mean()),
        ssl_label_correlation=corr)
    if conflicting:
        info.notes.append(
            f"{conflicting} rows share a feature vector with a row of the "
            f"opposite label (irreducible noise); they are kept.")

    if verbose:
        print(f"Loaded {path or 'dataframe'}: {raw_rows:,} rows")
        print(f"  label column '{label_col}', raw counts {raw_counts}")
        print(f"  encoding: {PHISHING_RAW_VALUE} -> phishing (1), otherwise legitimate (0)")
        print(f"  removed {duplicates_removed:,} duplicate rows -> {len(X):,} rows, "
              f"{X.shape[1]} features")
        print(f"  phishing rate {y.mean():.3f}")
        if corr is not None:
            print(f"  sanity check corr(SSLfinal_State, phishing) = {corr:+.3f}  (must be < 0) OK")
        for n in info.notes:
            print("  note:", n)
    return X, y, info


def load(path: str | None = None, verbose: bool = True):
    """Find, read and prepare the dataset in one call."""
    path = find_dataset(path)
    return prepare(load_raw(path), path=path, verbose=verbose)


In [ ]:
%%writefile phishdrift/drift.py
"""Simulated distribution shift for a corpus without timestamps.

The UCI corpus has no observation dates, so a "future" partition is built by
ranking records on an *atypicality* score and holding out the most atypical
ones. Atypicality is the mean absolute z-score over the most discriminative
features, so the held-out partition is shifted along exactly the dimensions
the detector relies on.

Two constructions are provided:

``naive_split``
    Takes the most atypical records overall. Because discriminative features
    separate the classes, this also moves the class prior - covariate shift
    and label shift are entangled. Kept only as a diagnostic.

``balanced_split``
    Applies the same ranking *within each class*, so the future partition is
    still the most atypical data but the class prior is held fixed. All
    reported drift results use this split.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split

from .config import (FUTURE_FRACTION, KS_ALPHA, N_DRIFT_FEATURES,
                     RANDOM_STATE, VAL_FRACTION)


@dataclass
class DriftSplit:
    name: str
    X_train: pd.DataFrame
    y_train: pd.Series
    X_val: pd.DataFrame
    y_val: pd.Series
    X_future: pd.DataFrame
    y_future: pd.Series
    old_pool_index: pd.Index

    def priors(self) -> dict:
        old = pd.concat([self.y_train, self.y_val])
        return {
            "old_pool": float(old.mean()),
            "train": float(self.y_train.mean()),
            "validation": float(self.y_val.mean()),
            "future": float(self.y_future.mean()),
            "prior_shift_pts": 100 * float(self.y_future.mean() - old.mean()),
            "n_train": len(self.y_train), "n_val": len(self.y_val),
            "n_future": len(self.y_future),
        }


def discriminative_features(X: pd.DataFrame, y: pd.Series, k: int = N_DRIFT_FEATURES) -> list:
    corr = X.apply(lambda c: abs(np.corrcoef(c, y)[0, 1]) if c.std() > 0 else 0.0)
    return corr.sort_values(ascending=False).head(k).index.tolist()


def atypicality(X: pd.DataFrame, features: list) -> pd.Series:
    Z = (X[features] - X[features].mean()) / X[features].std(ddof=0).replace(0, 1)
    return Z.abs().mean(axis=1)


def _finish(name, X, y, future_idx, rs) -> DriftSplit:
    old_idx = X.index.difference(future_idx)
    X_tr, X_va, y_tr, y_va = train_test_split(
        X.loc[old_idx], y.loc[old_idx], test_size=VAL_FRACTION,
        stratify=y.loc[old_idx], random_state=rs)
    return DriftSplit(name, X_tr, y_tr, X_va, y_va,
                      X.loc[future_idx], y.loc[future_idx], old_idx)


def naive_split(X, y, features, frac=FUTURE_FRACTION, rs=RANDOM_STATE) -> DriftSplit:
    score = atypicality(X, features)
    # stable tie-breaking so the split is reproducible
    order = score.sample(frac=1, random_state=rs).sort_values(ascending=False, kind="mergesort")
    future_idx = order.index[: int(round(frac * len(X)))]
    return _finish("naive", X, y, future_idx, rs)


def balanced_split(X, y, features, frac=FUTURE_FRACTION, rs=RANDOM_STATE) -> DriftSplit:
    score = atypicality(X, features)
    future = []
    for cls in (0, 1):
        s = score[y == cls].sample(frac=1, random_state=rs).sort_values(
            ascending=False, kind="mergesort")
        future.append(s.index[: int(round(frac * len(s)))])
    future_idx = future[0].append(future[1])
    return _finish("balanced", X, y, future_idx, rs)


def ks_tests(X_old: pd.DataFrame, X_new: pd.DataFrame, alpha=KS_ALPHA) -> pd.DataFrame:
    """Two-sample Kolmogorov-Smirnov test per feature, old pool vs future."""
    rows = []
    for f in X_old.columns:
        stat, p = ks_2samp(X_old[f], X_new[f])
        rows.append({"feature": f, "D": stat, "p_value": p, "shifted": p < alpha})
    return pd.DataFrame(rows).sort_values("D", ascending=False).reset_index(drop=True)


In [ ]:
%%writefile phishdrift/explain.py
"""SHAP attributions and explanation-stability metrics.

Stability is measured with a *fixed* model: the same fitted Random Forest is
explained on the in-distribution validation partition and on the shifted
future partition, so any change in attributions reflects the data, not
retraining.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from .config import RANDOM_STATE, SHAP_SAMPLE, TOP_K


def shap_values_positive(model, X: pd.DataFrame) -> np.ndarray:
    """SHAP values for the phishing class (label 1) from a tree model."""
    import shap
    vals = shap.TreeExplainer(model).shap_values(X)
    if isinstance(vals, list):
        vals = vals[1]
    elif np.ndim(vals) == 3:
        vals = vals[:, :, 1]
    return np.asarray(vals)


def sample(X: pd.DataFrame, n: int = SHAP_SAMPLE, rs: int = RANDOM_STATE) -> pd.DataFrame:
    return X.sample(min(n, len(X)), random_state=rs)


def attribution_summary(values: np.ndarray, X: pd.DataFrame) -> pd.DataFrame:
    """Per feature: mean |SHAP| (importance) and direction.

    Direction is the correlation between the feature's value and its SHAP
    value: positive means higher values push toward phishing. (The mean
    signed SHAP is not used - on a class-balanced sample it is close to zero
    by construction, so its sign is noise.)
    """
    Xv = np.asarray(X, dtype=float)
    direction = []
    for j in range(Xv.shape[1]):
        if Xv[:, j].std() == 0 or values[:, j].std() == 0:
            direction.append(0.0)
        else:
            direction.append(float(np.corrcoef(Xv[:, j], values[:, j])[0, 1]))
    return pd.DataFrame({"mean_abs": np.abs(values).mean(axis=0),
                         "direction": direction},
                        index=list(X.columns)).sort_values("mean_abs", ascending=False)


def stability(a: pd.DataFrame, b: pd.DataFrame, k: int = TOP_K) -> dict:
    """Compare two attribution summaries (same features, any order)."""
    b = b.reindex(a.index)
    rho = float(spearmanr(a["mean_abs"], b["mean_abs"]).correlation)
    top_a = set(a["mean_abs"].nlargest(k).index)
    top_b = set(b["mean_abs"].nlargest(k).index)
    jaccard = len(top_a & top_b) / len(top_a | top_b)
    flips = [f for f in a.index
             if np.sign(a.loc[f, "direction"]) * np.sign(b.loc[f, "direction"]) < 0
             and f in top_a]
    return {"spearman_rho": rho, f"jaccard_at_{k}": float(jaccard),
            "sign_flips_in_top_k": len(flips), "flipped_features": flips}


In [ ]:
%%writefile phishdrift/models.py
"""Base classifiers, the tuned Random Forest and the two hybrid ensembles.

Logistic Regression and the RBF SVM sit inside a ``Pipeline`` with a
``StandardScaler`` so the scaler is re-fitted inside every CV fold and never
sees held-out data. Tree models take raw features.
"""

from __future__ import annotations

import warnings

import pandas as pd
from sklearn.ensemble import (RandomForestClassifier, StackingClassifier,
                              VotingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold,
                                     cross_val_score, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from .config import CV_FOLDS, RANDOM_STATE, RF_GRID, TEST_SIZE

# scikit-learn >= 1.9 deprecates SVC(probability=True); it still works and keeps
# results comparable with older versions (e.g. the Kaggle image), so silence it.
warnings.filterwarnings("ignore", message=".*probability.*deprecated", category=FutureWarning)


def scaled(estimator) -> Pipeline:
    return Pipeline([("scaler", StandardScaler()), ("clf", estimator)])


def make_cv(rs: int = RANDOM_STATE) -> StratifiedKFold:
    return StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=rs)


def base_models(rs: int = RANDOM_STATE) -> dict:
    return {
        "Decision Tree": DecisionTreeClassifier(random_state=rs),
        "Random Forest": RandomForestClassifier(n_estimators=300, random_state=rs, n_jobs=-1),
        "Logistic Regression": scaled(LogisticRegression(max_iter=2000, random_state=rs)),
        "SVM (RBF)": scaled(SVC(kernel="rbf", probability=True, random_state=rs)),
    }


def ensemble_members(rs: int = RANDOM_STATE) -> list:
    return [
        ("rf", RandomForestClassifier(n_estimators=300, random_state=rs, n_jobs=-1)),
        ("lr", scaled(LogisticRegression(max_iter=2000, random_state=rs))),
        ("svm", scaled(SVC(kernel="rbf", probability=True, random_state=rs))),
    ]


def ensembles(rs: int = RANDOM_STATE) -> dict:
    return {
        "Soft-Voting Ensemble": VotingClassifier(ensemble_members(rs), voting="soft", n_jobs=-1),
        "Stacking Ensemble": StackingClassifier(
            estimators=ensemble_members(rs),
            final_estimator=LogisticRegression(max_iter=2000, random_state=rs),
            cv=CV_FOLDS, n_jobs=-1),
    }


def tune_random_forest(X, y, cv, rs: int = RANDOM_STATE) -> GridSearchCV:
    grid = GridSearchCV(RandomForestClassifier(random_state=rs, n_jobs=-1),
                        RF_GRID, cv=cv, scoring="f1", n_jobs=-1)
    grid.fit(X, y)
    return grid


def metrics(y_true, y_pred, y_prob) -> dict:
    """Positive class = phishing (label 1)."""
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
    }


def evaluate(name, model, X_train, y_train, X_test, y_test, cv):
    """Cross-validated F1 on train, then fit and score on the held-out test set."""
    cv_f1 = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1).mean()
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    row = {"Model": name, "CV F1": cv_f1, **metrics(y_test, pred, prob)}
    return row, model


def run_static(X, y, rs: int = RANDOM_STATE, verbose: bool = True):
    """The conventional experiment: 80/20 stratified split, 7 models."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=rs)
    cv = make_cv(rs)
    rows, fitted = [], {}

    def log(row):
        if verbose:
            print(f"  {row['Model']:24s} F1={row['F1']:.4f}  CV-F1={row['CV F1']:.4f}  "
                  f"AUC={row['ROC-AUC']:.4f}")

    for name, model in base_models(rs).items():
        row, fit = evaluate(name, model, X_train, y_train, X_test, y_test, cv)
        rows.append(row); fitted[name] = fit; log(row)

    grid = tune_random_forest(X_train, y_train, cv, rs)
    if verbose:
        print(f"  RF grid search best {grid.best_params_}  CV-F1 {grid.best_score_:.4f}")
    row, fit = evaluate("Random Forest (tuned)", grid.best_estimator_,
                        X_train, y_train, X_test, y_test, cv)
    rows.append(row); fitted[row["Model"]] = fit; log(row)

    for name, model in ensembles(rs).items():
        row, fit = evaluate(name, model, X_train, y_train, X_test, y_test, cv)
        rows.append(row); fitted[name] = fit; log(row)

    results = (pd.DataFrame(rows).set_index("Model")
               .sort_values("F1", ascending=False).round(4))
    split = {"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test}
    return results, fitted, split, grid.best_params_


In [ ]:
%%writefile phishdrift/pipeline.py
"""End-to-end experiment: static study, then the drift study.

Both the command-line script (``scripts/run_all.py``) and the Kaggle notebook
call these two functions, so the numbers in the README, the notebook and the
paper all come from the same code path.
"""

from __future__ import annotations

import os

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

from . import abstention as ab
from . import drift as dr
from . import explain as ex
from . import plots
from .config import RANDOM_STATE, TOP_K
from .models import run_static


def _r(x, n=4):
    return None if x is None or (isinstance(x, float) and np.isnan(x)) else round(float(x), n)


# --------------------------------------------------------------------------- #
# 1. Static study
# --------------------------------------------------------------------------- #
def static_study(X, y, fig_dir="figures", res_dir="results", verbose=True):
    os.makedirs(res_dir, exist_ok=True)
    if verbose:
        print("\n== Static study: stratified 80/20 split ==")
    plots.class_balance(y, fig_dir)
    plots.label_correlation(X, y, fig_dir)

    results, fitted, split, best_params = run_static(X, y, verbose=verbose)
    results.to_csv(os.path.join(res_dir, "static_results.csv"))
    plots.model_comparison(results, fig_dir)

    best_name = results.index[0]
    Xte, yte = split["X_test"], split["y_test"]
    prob = fitted[best_name].predict_proba(Xte)[:, 1]
    plots.confusion_and_roc(best_name, yte, prob, fig_dir)
    plots.pr_curve(best_name, yte, prob, fig_dir)

    rf = fitted["Random Forest (tuned)"]
    imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
    imp.to_csv(os.path.join(res_dir, "rf_importance.csv"), header=["importance"])
    plots.importance_bar(imp, "Random Forest impurity importance", "mean decrease in impurity",
                         fig_dir, "rf_importance.png")

    perm = permutation_importance(fitted[best_name], Xte, yte, scoring="f1",
                                  n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
    perm_s = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
    perm_s.to_csv(os.path.join(res_dir, "permutation_importance.csv"), header=["f1_drop"])
    plots.importance_bar(perm_s, f"Permutation importance - {best_name}",
                         "drop in F1 when the feature is shuffled", fig_dir,
                         "permutation_importance.png", color=plots.ORANGE)

    Xs = ex.sample(Xte)
    sv = ex.shap_values_positive(rf, Xs)
    plots.shap_beeswarm(sv, Xs, fig_dir)
    shap_top = ex.attribution_summary(sv, Xs)
    shap_top.to_csv(os.path.join(res_dir, "shap_static.csv"))

    return {
        "results": results,
        "best_model": best_name,
        "best_rf_params": best_params,
        "top_features_shap": shap_top.index[:TOP_K].tolist(),
        "top_features_permutation": perm_s.index[:TOP_K].tolist(),
    }


# --------------------------------------------------------------------------- #
# 2. Drift study
# --------------------------------------------------------------------------- #
def drift_study(X, y, fig_dir="figures", res_dir="results", verbose=True):
    os.makedirs(res_dir, exist_ok=True)
    if verbose:
        print("\n== Drift study: simulated shift, class prior held fixed ==")

    feats = dr.discriminative_features(X, y)
    naive = dr.naive_split(X, y, feats)
    bal = dr.balanced_split(X, y, feats)
    pn, pb = naive.priors(), bal.priors()
    plots.drift_priors(pn, pb, fig_dir)
    if verbose:
        print(f"  naive split    phishing rate old {pn['old_pool']:.3f} -> future {pn['future']:.3f}"
              f"  (label shift {pn['prior_shift_pts']:+.1f} pts, diagnostic only)")
        print(f"  balanced split phishing rate old {pb['old_pool']:.3f} -> future {pb['future']:.3f}"
              f"  ({pb['n_train']} train / {pb['n_val']} val / {pb['n_future']} future)")

    X_old = X.loc[bal.old_pool_index]
    ks = dr.ks_tests(X_old, bal.X_future)
    ks.to_csv(os.path.join(res_dir, "ks_tests.csv"), index=False)
    plots.ks_bar(ks, fig_dir)
    n_shift = int(ks["shifted"].sum())

    # Base detector: Random Forest trained on the old pool's training part only.
    rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(bal.X_train, bal.y_train)
    p_val_raw = rf.predict_proba(bal.X_val)[:, 1]
    p_fut_raw = rf.predict_proba(bal.X_future)[:, 1]

    cal = ab.fit_calibrated(
        RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
        bal.X_train, bal.y_train)
    p_val = cal.predict_proba(bal.X_val)[:, 1]
    p_fut = cal.predict_proba(bal.X_future)[:, 1]

    calib = {
        "uncalibrated_validation": ab.calibration_metrics(bal.y_val, p_val_raw),
        "uncalibrated_future": ab.calibration_metrics(bal.y_future, p_fut_raw),
        "calibrated_validation": ab.calibration_metrics(bal.y_val, p_val),
        "calibrated_future": ab.calibration_metrics(bal.y_future, p_fut),
    }
    plots.reliability(bal.y_val, p_val, bal.y_future, p_fut, fig_dir)

    tau = ab.select_threshold(bal.y_val, p_val)     # locked on validation only
    sel_val = ab.selective_metrics(bal.y_val, p_val, tau)
    sel_fut = ab.selective_metrics(bal.y_future, p_fut, tau)
    rc_val, rc_fut = ab.risk_coverage(bal.y_val, p_val), ab.risk_coverage(bal.y_future, p_fut)
    pd.concat([rc_val.assign(partition="validation"), rc_fut.assign(partition="future")]) \
        .to_csv(os.path.join(res_dir, "risk_coverage.csv"), index=False)
    plots.risk_coverage(rc_val, rc_fut, sel_fut["coverage"], fig_dir)
    plots.drift_accuracy(sel_val["accuracy_all"], sel_fut["accuracy_all"],
                         sel_fut["selective_accuracy"], sel_fut["coverage"], fig_dir)
    bins = ab.confidence_bins(bal.y_future, p_fut)
    bins.to_csv(os.path.join(res_dir, "confidence_bins_future.csv"), index=False)
    plots.confidence_bins_plot(bins, fig_dir)

    # Explanation stability: same fitted model, two partitions.
    Xv_s, Xf_s = ex.sample(bal.X_val), ex.sample(bal.X_future)
    s_val = ex.attribution_summary(ex.shap_values_positive(rf, Xv_s), Xv_s)
    s_fut = ex.attribution_summary(ex.shap_values_positive(rf, Xf_s), Xf_s)
    stab = ex.stability(s_val, s_fut)
    s_val.join(s_fut, lsuffix="_val", rsuffix="_future") \
        .to_csv(os.path.join(res_dir, "shap_stability.csv"))
    plots.shap_stability(s_val, s_fut, stab, fig_dir)

    if verbose:
        print(f"  KS: {n_shift}/{len(ks)} features shifted (p < 0.05)")
        print(f"  accuracy validation {sel_val['accuracy_all']:.4f} -> future {sel_fut['accuracy_all']:.4f}")
        print(f"  tau locked on validation = {tau:.2f}; future coverage {sel_fut['coverage']:.4f}, "
              f"selective accuracy {sel_fut['selective_accuracy']:.4f}")
        print(f"  calibrated future Brier {calib['calibrated_future']['brier']:.4f}, "
              f"ECE {calib['calibrated_future']['ece']:.4f}")
        print(f"  SHAP stability Spearman {stab['spearman_rho']:.4f}, "
              f"Jaccard@{TOP_K} {stab[f'jaccard_at_{TOP_K}']:.2f}, "
              f"sign flips in top-{TOP_K}: {stab['sign_flips_in_top_k']}")

    return {
        "discriminative_features": feats,
        "naive_priors": {k: _r(v) for k, v in pn.items()},
        "balanced_priors": {k: _r(v) for k, v in pb.items()},
        "ks_shifted": n_shift, "ks_total": len(ks),
        "threshold": tau,
        "validation": {k: _r(v) for k, v in sel_val.items()},
        "future": {k: _r(v) for k, v in sel_fut.items()},
        "calibration": {k: {m: _r(v) for m, v in d.items()} for k, d in calib.items()},
        "confidence_bins_future": bins.round(4).to_dict(orient="records"),
        "shap_stability": {k: (_r(v) if isinstance(v, float) else v) for k, v in stab.items()},
    }


In [ ]:
%%writefile phishdrift/plots.py
"""Every figure used in the README and the paper.

Static PNGs, one value axis per chart, a fixed categorical order
(``config.SERIES``), a legend whenever there are two or more series, and
labels in ink rather than series colour.
"""

from __future__ import annotations

import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, precision_recall_curve, roc_curve

from .config import (AQUA, BLUE, CLASS_NAMES, GRID, INK, MUTED, ORANGE,
                     SEQ_CMAP, SERIES)

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 160, "font.size": 10,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "axes.titleweight": "bold", "axes.titlesize": 11,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.axisbelow": True, "legend.frameon": False,
})


def _save(fig, out_dir, name):
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, name)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return path


def class_balance(y, out_dir):
    fig, ax = plt.subplots(figsize=(4.6, 3.4))
    counts = np.bincount(np.asarray(y), minlength=2)
    bars = ax.bar(CLASS_NAMES, counts, color=[BLUE, ORANGE], width=0.55)
    for b, v in zip(bars, counts):
        ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,}\n({v / counts.sum():.1%})",
                ha="center", va="bottom", color=INK, fontsize=9)
    ax.set_ylim(0, counts.max() * 1.25)
    ax.set_ylabel("sites")
    ax.set_title("Class balance after deduplication")
    ax.grid(axis="x", visible=False)
    return _save(fig, out_dir, "class_balance.png")


def label_correlation(X, y, out_dir, k=12):
    corr = X.apply(lambda c: np.corrcoef(c, y)[0, 1]).dropna()
    corr = corr.reindex(corr.abs().sort_values(ascending=False).index).head(k)[::-1]
    fig, ax = plt.subplots(figsize=(6.4, 4.4))
    ax.barh(corr.index, corr.values,
            color=[ORANGE if v > 0 else BLUE for v in corr.values], height=0.6)
    ax.axvline(0, color=MUTED, lw=0.8)
    ax.set_xlabel("correlation with phishing label  (+ pushes toward phishing)")
    ax.set_title(f"Top {k} features by correlation with the label")
    ax.grid(axis="y", visible=False)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=ORANGE, label="toward phishing"),
                       Patch(color=BLUE, label="toward legitimate")],
              loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2)
    return _save(fig, out_dir, "label_correlation.png")


def model_comparison(results, out_dir):
    cols = ["F1", "ROC-AUC", "CV F1"]
    df = results[cols]
    x = np.arange(len(df))
    w = 0.26
    fig, ax = plt.subplots(figsize=(10, 4.8))
    for i, c in enumerate(cols):
        ax.bar(x + (i - 1) * w, df[c], w - 0.03, color=SERIES[i], label=c)
    ax.set_xticks(x, df.index, rotation=20, ha="right")
    ax.set_ylim(max(0, df.values.min() - 0.05), 1.0)
    ax.set_ylabel("score")
    ax.set_title("Model comparison on the conventional 80/20 split")
    ax.legend(ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.0))
    ax.set_title("Model comparison on the conventional 80/20 split", pad=28)
    ax.grid(axis="x", visible=False)
    return _save(fig, out_dir, "model_comparison.png")


def confusion_and_roc(name, y_true, prob, out_dir):
    pred = (prob >= 0.5).astype(int)
    cm = confusion_matrix(y_true, pred)
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    ax = axes[0]
    ax.imshow(cm, cmap=SEQ_CMAP)
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, f"{v:,}", ha="center", va="center",
                color="white" if v > cm.max() / 2 else INK, fontsize=12)
    ax.set_xticks([0, 1], CLASS_NAMES); ax.set_yticks([0, 1], CLASS_NAMES)
    ax.set_xlabel("predicted"); ax.set_ylabel("actual"); ax.grid(False)
    ax.set_title(f"Confusion matrix - {name}")
    fpr, tpr, _ = roc_curve(y_true, prob)
    from sklearn.metrics import roc_auc_score
    axes[1].plot(fpr, tpr, color=BLUE, lw=2, label=f"{name} (AUC {roc_auc_score(y_true, prob):.4f})")
    axes[1].plot([0, 1], [0, 1], ls="--", color=MUTED, lw=1, label="chance")
    axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate")
    axes[1].set_title("ROC curve"); axes[1].legend(loc="lower right")
    return _save(fig, out_dir, "confusion_roc.png")


def pr_curve(name, y_true, prob, out_dir):
    from sklearn.metrics import average_precision_score
    p, r, _ = precision_recall_curve(y_true, prob)
    fig, ax = plt.subplots(figsize=(5.6, 4.2))
    ax.plot(r, p, color=BLUE, lw=2, label=f"{name} (AP {average_precision_score(y_true, prob):.4f})")
    ax.axhline(np.mean(y_true), ls="--", color=MUTED, lw=1, label="no-skill baseline")
    ax.set_xlabel("recall (phishing)"); ax.set_ylabel("precision (phishing)")
    ax.set_ylim(0, 1.02); ax.set_title("Precision-recall curve"); ax.legend(loc="lower left")
    return _save(fig, out_dir, "precision_recall.png")


def importance_bar(series, title, xlabel, out_dir, fname, color=BLUE, k=15):
    s = series.head(k)[::-1]
    fig, ax = plt.subplots(figsize=(7, 4.8))
    ax.barh(s.index, s.values, color=color, height=0.6)
    ax.set_xlabel(xlabel); ax.set_title(title); ax.grid(axis="y", visible=False)
    return _save(fig, out_dir, fname)


def shap_beeswarm(values, X, out_dir, fname="shap_summary.png",
                  title="SHAP - impact of each feature on the phishing prediction"):
    import shap
    shap.summary_plot(values, X, max_display=15, show=False)
    fig = plt.gcf()
    fig.axes[0].set_title(title)
    return _save(fig, out_dir, fname)


def drift_priors(naive_priors, balanced_priors, out_dir):
    labels = ["old pool", "future"]
    fig, ax = plt.subplots(figsize=(6, 3.8))
    x = np.arange(2); w = 0.34
    for i, (name, pr) in enumerate([("naive split", naive_priors),
                                    ("class-balanced split", balanced_priors)]):
        vals = [100 * pr["old_pool"], 100 * pr["future"]]
        bars = ax.bar(x + (i - 0.5) * w, vals, w - 0.04, color=SERIES[i], label=name)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.1f}%",
                    ha="center", color=INK, fontsize=9)
    ax.set_xticks(x, labels); ax.set_ylim(0, 110)
    ax.set_ylabel("phishing rate (%)")
    ax.set_title("Class prior per partition: the naive split moves it")
    ax.legend(loc="upper left"); ax.grid(axis="x", visible=False)
    return _save(fig, out_dir, "drift_priors.png")


def ks_bar(ks, out_dir, k=15):
    s = ks.head(k)[::-1]
    fig, ax = plt.subplots(figsize=(7, 4.8))
    ax.barh(s["feature"], s["D"], height=0.6,
            color=[ORANGE if sh else BLUE for sh in s["shifted"]])
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=ORANGE, label="shifted (p < 0.05)"),
                       Patch(color=BLUE, label="not significant")], loc="lower right")
    ax.set_xlabel("KS statistic D (old pool vs future)")
    ax.set_title(f"Feature shift under the balanced split - {int(ks['shifted'].sum())} of {len(ks)} shifted")
    ax.grid(axis="y", visible=False)
    return _save(fig, out_dir, "ks_shift.png")


def drift_accuracy(val_acc, fut_acc, sel_acc, coverage, out_dir):
    names = ["validation\n(in-distribution)", "future\n(shifted)",
             f"future, answered\n({coverage:.0%} coverage)"]
    vals = [100 * val_acc, 100 * fut_acc, 100 * sel_acc]
    fig, ax = plt.subplots(figsize=(6.4, 4))
    bars = ax.bar(names, vals, color=[BLUE, ORANGE, AQUA], width=0.55)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.2f}%", ha="center", color=INK)
    ax.set_ylim(0, 110); ax.set_ylabel("accuracy (%)")
    ax.set_title("Accuracy under drift, and what abstention recovers")
    ax.grid(axis="x", visible=False)
    return _save(fig, out_dir, "drift_accuracy.png")


def reliability(y_val, p_val, y_fut, p_fut, out_dir, n_bins=10):
    from sklearn.calibration import calibration_curve
    fig, ax = plt.subplots(figsize=(5.4, 4.6))
    ax.plot([0, 1], [0, 1], ls="--", color=MUTED, lw=1, label="perfect calibration")
    for (y, p, name, c) in [(y_val, p_val, "validation", BLUE), (y_fut, p_fut, "future", ORANGE)]:
        frac, mean = calibration_curve(y, p, n_bins=n_bins, strategy="quantile")
        ax.plot(mean, frac, marker="o", ms=5, lw=2, color=c, label=name)
    ax.set_xlabel("mean predicted P(phishing)"); ax.set_ylabel("observed phishing rate")
    ax.set_title("Reliability diagram (calibrated model)"); ax.legend(loc="upper left")
    return _save(fig, out_dir, "reliability.png")


def risk_coverage(rc_val, rc_fut, tau_cov_fut, out_dir):
    fig, ax = plt.subplots(figsize=(6, 4.2))
    ax.plot(100 * rc_val["coverage"], 100 * rc_val["risk"], color=BLUE, lw=2, label="validation")
    ax.plot(100 * rc_fut["coverage"], 100 * rc_fut["risk"], color=ORANGE, lw=2, label="future")
    ax.axvline(100 * tau_cov_fut, color=MUTED, ls=":", lw=1.2, label="locked threshold (future)")
    ax.set_xlabel("coverage (% of cases answered)"); ax.set_ylabel("error rate on answered (%)")
    ax.set_title("Risk-coverage curve"); ax.legend(loc="upper left")
    return _save(fig, out_dir, "risk_coverage.png")


def confidence_bins_plot(bins, out_dir):
    fig, ax = plt.subplots(figsize=(6.4, 4))
    bars = ax.bar(bins["bin"], 100 * bins["accuracy"].fillna(0), color=BLUE, width=0.6)
    for b, (acc, n) in zip(bars, zip(bins["accuracy"], bins["n"])):
        if n:
            ax.text(b.get_x() + b.get_width() / 2, 100 * acc + 1, f"{acc:.0%}\nn={n}",
                    ha="center", color=INK, fontsize=8)
    ax.set_ylim(0, 115); ax.set_xlabel("calibrated confidence max(p, 1-p)")
    ax.set_ylabel("accuracy on future partition (%)")
    ax.set_title("Is confidence still diagnostic under drift?")
    ax.grid(axis="x", visible=False)
    return _save(fig, out_dir, "confidence_bins.png")


def shap_stability(a, b, stab, out_dir):
    b = b.reindex(a.index)
    fig, ax = plt.subplots(figsize=(5.4, 5))
    ax.scatter(a["mean_abs"], b["mean_abs"], s=40, color=BLUE,
               edgecolor="white", linewidth=1.5, zorder=3)
    lim = 1.05 * max(a["mean_abs"].max(), b["mean_abs"].max())
    ax.plot([0, lim], [0, lim], ls="--", color=MUTED, lw=1)
    for f in a.index[:5]:
        ax.annotate(f, (a.loc[f, "mean_abs"], b.loc[f, "mean_abs"]), fontsize=8,
                    color=INK, xytext=(4, 4), textcoords="offset points")
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel("mean |SHAP| - validation"); ax.set_ylabel("mean |SHAP| - future")
    k = [x for x in stab if x.startswith("jaccard")][0]
    ax.set_title(f"Explanation stability  (Spearman {stab['spearman_rho']:.3f}, "
                 f"Jaccard@{k.split('_')[-1]} {stab[k]:.2f})")
    return _save(fig, out_dir, "shap_stability.png")


## 1. Load and clean the data

In [ ]:
import warnings, json
warnings.filterwarnings('ignore')
import pandas as pd
from IPython.display import Image, display, Markdown
from phishdrift.data import load
from phishdrift.pipeline import static_study, drift_study

X, y, info = load()          # pass a path here if auto-detection picks the wrong file
FIG, RES = 'figures', 'results'

## 2. Static study

In [ ]:
st = static_study(X, y, FIG, RES)
st['results']

In [ ]:
for f in ['class_balance', 'label_correlation', 'model_comparison', 'confusion_roc',
          'precision_recall', 'rf_importance', 'permutation_importance', 'shap_summary']:
    display(Image(f'{FIG}/{f}.png'))

## 3. Drift study

The UCI corpus has no timestamps, so a "future" partition is simulated: records are ranked by atypicality (mean |z| over the 10 most discriminative features) and the most atypical 30% *of each class* are held out. Ranking within each class keeps the class prior fixed, so the shift is covariate shift rather than label shift. The naive version (top 30% overall) is shown only to demonstrate how it entangles the two.

The model is a Random Forest trained on the old pool, calibrated with 5-fold isotonic regression. The abstention threshold is chosen on validation data and locked before the future partition is scored.

In [ ]:
dr = drift_study(X, y, FIG, RES)

In [ ]:
for f in ['drift_priors', 'ks_shift', 'drift_accuracy', 'reliability',
          'risk_coverage', 'confidence_bins', 'shap_stability']:
    display(Image(f'{FIG}/{f}.png'))

## 4. Summary

In [ ]:
import dataclasses
v, fu, s = dr['validation'], dr['future'], dr['shap_stability']
cal = dr['calibration']
rows = {
    'Best static model': st['best_model'],
    'Best static F1 (phishing)': st['results'].iloc[0]['F1'],
    'Balanced split phishing rate old -> future': f"{dr['balanced_priors']['old_pool']:.3f} -> {dr['balanced_priors']['future']:.3f}",
    'Features shifted (KS, p<0.05)': f"{dr['ks_shifted']} / {dr['ks_total']}",
    'Accuracy validation -> future': f"{v['accuracy_all']:.4f} -> {fu['accuracy_all']:.4f}",
    'Threshold (locked on validation)': dr['threshold'],
    'Future coverage': fu['coverage'],
    'Future selective accuracy': fu['selective_accuracy'],
    'Future Brier (calibrated)': cal['calibrated_future']['brier'],
    'Future ECE (calibrated)': cal['calibrated_future']['ece'],
    'SHAP Spearman (val vs future)': s['spearman_rho'],
    'SHAP Jaccard@10': s['jaccard_at_10'],
}
summary = pd.Series(rows, name='value').to_frame()
summary

In [ ]:
with open(f'{RES}/summary.json', 'w') as fh:
    json.dump({'data': dataclasses.asdict(info), 'best_rf_params': st['best_rf_params'],
               'static_results': st['results'].reset_index().to_dict(orient='records'),
               'drift': dr}, fh, indent=2, default=str)
print('Saved results/ and figures/ - download them from the Output panel.')